# Documentação técnica — Trilha SEFAZ-SP

Este notebook descreve arquitetura, módulos, fluxo dos dados, autenticação, testes, operação e manutenção do projeto. As células executáveis inspecionam somente código e metadados; não abrem materiais de estudo nem mostram valores de configuração.

## Objetivos

- Entender como HTML, CSS, JavaScript, Python e Supabase se conectam.
- Documentar a transformação dos arquivos Markdown nos JSONs consumidos pelo site.
- Orientar configuração local, testes e publicação.
- Executar verificações seguras do código e das políticas, sem imprimir dados privados.

> **Privacidade:** os materiais por disciplina e `data/` são ignorados pelo Git. Não publique conteúdo didático, senhas, tokens ou chaves privadas. A chave Supabase publicável é destinada ao navegador; nunca inclua `service_role`, `sb_secret_` ou credenciais equivalentes.

Execute as células Python em ordem, usando um kernel Python local.

In [ ]:
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "extractor.py").is_file() and (path / "app.js").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abra este notebook a partir da pasta que contém app.js e extractor.py.")

CORE_FILES = (
    "index.html", "styles.css", "app.js", "extractor.py",
    "supabase-config.js", "supabase-schema.sql", "README.md",
)
missing_files = [name for name in CORE_FILES if not (PROJECT_ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Arquivos principais ausentes: {', '.join(missing_files)}")

print(f"Raiz do projeto: {PROJECT_ROOT}")
print("Arquivos principais verificados:")
for name in CORE_FILES:
    print(f"  - {name}")

## 1. Visão geral e arquitetura

A Trilha é uma aplicação estática para estudos. O navegador entrega interface e interações; um script Python prepara os materiais localmente; o Supabase fornece autenticação, armazenamento privado e sincronização do progresso.

```text
Fontes Markdown locais
        │
        ▼
extractor.py ──► manifest.json + subjects/*.json + question-index.json
                                      │ (upload manual para bucket privado)
                                      ▼
index.html + styles.css + app.js ──► Supabase Auth / Storage / user_progress
                                      └──► localStorage (progresso e preferências)
```

### Áreas funcionais

- **Visão geral:** métricas de desempenho e atividade de estudo.
- **Conteúdos:** leitor Markdown, busca e navegação por material, marcação de estudo.
- **Questões:** treino e simulados com alternativas, gabaritos, comentários e histórico.
- **Mapas mentais e flashcards:** recursos de revisão derivados dos materiais.
- **Organização dos estudos:** plano, calendário, metas, Pomodoro, revisões e diário.
- **Conta e histórico:** acesso restrito, recuperação de senha e sincronização.

| Arquivo | Responsabilidade |
|---|---|
| `index.html` | Estrutura da página, navegação, tela de autenticação e ponto de montagem das telas. |
| `styles.css` | Layout, componentes visuais, leitor e comportamento responsivo. |
| `app.js` | Estado da interface, autenticação, leitura de dados, renderização e interações. |
| `extractor.py` | Limpeza dos Markdown, extração de questões/flashcards e geração de JSONs. |
| `tests/test_extractor.py` | Testes unitários da extração e normalização. |
| `supabase-config.js` | URL pública, chave publishable e nome do bucket. |
| `supabase-schema.sql` | Tabela de progresso, bucket privado e políticas RLS. |
| `README.md` / `.gitignore` | Instruções operacionais / exclusão de dados locais e privados. |

A próxima célula lista assinaturas e contagens de código; não imprime código-fonte.

In [ ]:
import ast
import re

source_paths = {
    "HTML": PROJECT_ROOT / "index.html",
    "CSS": PROJECT_ROOT / "styles.css",
    "JavaScript": PROJECT_ROOT / "app.js",
    "Python": PROJECT_ROOT / "extractor.py",
    "SQL": PROJECT_ROOT / "supabase-schema.sql",
}
for label, path in source_paths.items():
    print(f"{label:12} {len(path.read_text(encoding='utf-8').splitlines()):5} linhas")

app_source = source_paths["JavaScript"].read_text(encoding="utf-8")
functions = re.findall(
    r"(?m)^\s*(?:async\s+)?function\s+(initializeAuth|loadPrivateJson|loadSubject|setTab|renderDashboard|renderContents|renderQuestions|renderOrganization|renderMindmaps|renderFlashcards)\s*\(",
    app_source,
)
print("\nFunções principais encontradas:")
for name in dict.fromkeys(functions):
    print(f"  - {name}")

python_tree = ast.parse((PROJECT_ROOT / "extractor.py").read_text(encoding="utf-8"))
extractor_functions = [node.name for node in python_tree.body if isinstance(node, ast.FunctionDef)]
print(f"\nFunções do extrator: {', '.join(extractor_functions)}")
test_tree = ast.parse((PROJECT_ROOT / "tests" / "test_extractor.py").read_text(encoding="utf-8"))
test_count = sum(isinstance(node, ast.FunctionDef) and node.name.startswith("test_") for node in ast.walk(test_tree))
print(f"Casos de teste declarados: {test_count}")

## 2. Pipeline dos materiais e contrato dos dados

`extractor.py` usa somente a biblioteca padrão do Python. `extract()` considera as subpastas imediatas da origem como disciplinas e procura `.md` recursivamente em cada uma.

1. Lê UTF-8 (com tolerância a BOM e caracteres inválidos).
2. Separa teoria de questões de múltipla escolha e tenta identificar assunto, banca, alternativas, gabarito e comentário.
3. Limpa contadores, cabeçalhos repetidos, marcação residual e publicidade de conversão PDF/OCR.
4. Deduplica por identificador externo dentro da disciplina, preservando a ocorrência mais completa.
5. Sugere flashcards a partir de títulos e parágrafos; revise-os antes de tratá-los como fonte definitiva.
6. Grava um JSON por disciplina, o manifesto e o índice leve de questões.

### Arquivos e campos

- `manifest.json`: data de geração, disciplinas, materiais e totais.
- `subjects/<id>.json`: objeto `subject` e lista `materials`.
- Cada material: `id`, `title`, `source`, `content`, `questions` e `flashcards`.
- Cada questão: `id`, `number`, `statement`, `options`, `answer`, `solution`, `source`, `topic` e `bank`.
- `question-index.json`: associa questão a `subjectId`, `topic` e `bank` sem baixar todos os materiais.

Não faça commit de `data/` ou dos arquivos de materiais. Envie os JSONs ao bucket privado `study-materials`, mantendo os caminhos do manifesto.

```powershell
# Gera dados a partir das pastas de materiais do projeto
python extractor.py

# Origem e destino alternativos
python extractor.py "C:\\caminho\\para\\materiais" --output ".\\data"
```

A célula abaixo exercita o parser em memória, sem escrever arquivos ou ler materiais reais.

In [ ]:
import sys

project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
from extractor import extract_questions

example_markdown = """## Noções básicas\n\nQuestão 12345\nEnunciado demonstrativo.\nA) Alternativa um\nB) Alternativa dois\nGabarito: B\n"""
theory, parsed_questions = extract_questions(example_markdown, "Exemplo/aula.md")
assert len(parsed_questions) == 1
assert [option["key"] for option in parsed_questions[0]["options"]] == ["A", "B"]
assert parsed_questions[0]["answer"] == "B"
assert "Enunciado demonstrativo" not in theory
print("Parser em memória validado: uma questão, duas alternativas e gabarito identificado.")
print("Nenhum arquivo de saída foi criado nesta verificação.")

## 3. Aplicação web: navegação, renderização e estado

### Inicialização e carregamento

1. `index.html` define o shell e a tela de autenticação; scripts carregam com `defer`.
2. `app.js` inicia em `start()`, prepara o cliente Supabase e acompanha a sessão.
3. Após autenticação e leitura do progresso, `loadStudyMaterials()` busca `manifest.json` do bucket.
4. A navegação é montada pelo manifesto; `loadSubject()` baixa o JSON de uma disciplina sob demanda e o mantém em cache de memória.
5. `setTab()` delega a tela para funções `render*`, que conectam seus eventos.

O manifesto é um índice pequeno; materiais maiores são buscados quando necessários. Falhas de rede e de autenticação devem ser explícitas, sem renderizar sucesso falso.

### Responsabilidades do JavaScript

- **Estado:** estado transitório em memória; histórico e preferências locais em `localStorage`, com chaves versionadas.
- **Autenticação/sincronização:** restringe a conta, carrega `user_progress` e agrupa salvamentos. A importação do histórico local é explícita.
- **Conteúdos:** normaliza Markdown e artefatos PDF/OCR; renderiza títulos, listas, tabelas, sumários e chamadas visuais. Escapa texto HTML e restringe protocolos de links.
- **Questões:** registra respostas, correção, duração, alternativas eliminadas e resultados de simulados.
- **Demais áreas:** dashboard, mapas mentais, flashcards e organização dos estudos.

`styles.css` organiza os componentes por área e aplica breakpoints responsivos. `index.html` mantém navegação e elementos permanentes; `app.js` injeta a tela ativa em `#view`.

A célula verifica as seis áreas da navegação e a existência de breakpoints CSS.

In [ ]:
html_source = (PROJECT_ROOT / "index.html").read_text(encoding="utf-8")
css_source = (PROJECT_ROOT / "styles.css").read_text(encoding="utf-8")
nav_tabs = re.findall(r'<button class="nav-link[^\"]*" data-tab="([^"]+)"', html_source)
expected_tabs = {"dashboard", "contents", "questions", "mindmaps", "flashcards", "organization"}
assert set(nav_tabs) == expected_tabs, f"Navegação inesperada: {nav_tabs}"
assert len(nav_tabs) == len(set(nav_tabs)), "Há áreas duplicadas na navegação."
breakpoints = re.findall(r"@media\s*\(max-width:\s*([^)]+)\)", css_source)
assert breakpoints, "Nenhuma regra responsiva @media foi encontrada."
print(f"Áreas verificadas: {len(nav_tabs)} — {', '.join(nav_tabs)}")
print(f"Breakpoints responsivos CSS: {', '.join(breakpoints)}")

## 4. Supabase, autenticação e privacidade

A aplicação é estática: não possui servidor próprio. `supabase-config.js` contém somente URL, chave **publishable** e bucket; isso conecta o cliente, mas não substitui autorização.

- `app.js` valida a conta permitida, busca manifesto/materiais no Storage e sincroniza o progresso em `user_progress`.
- `supabase-schema.sql` habilita RLS em `user_progress`, limita operações à conta autorizada e configura o bucket privado com limite por arquivo.
- O progresso local continua em `localStorage`; a interface permite importação explícita e informa falhas de sincronização.
- Identificadores da conta presentes no cliente e nas políticas precisam permanecer coerentes. Alterar a conta exige revisar ambos e as políticas já implantadas.

### Checklist de configuração

1. Execute `supabase-schema.sql` no SQL Editor.
2. Crie a conta autorizada e desative novos cadastros públicos.
3. Configure URLs de redirecionamento para o site e para o servidor local.
4. Confira `url`, `publishableKey` e `materialsBucket`; não salve senhas, `service_role`, `sb_secret_` ou tokens privados no código/notebook.
5. Envie manifesto, índice e JSONs de disciplina ao bucket `study-materials`, que deve permanecer privado.
6. Publique apenas os arquivos estáticos necessários; nunca publique `data/` ou os materiais.

A checagem do cliente melhora a experiência; a autorização efetiva deve continuar nas políticas RLS do banco e do Storage.

In [ ]:
schema_sql = (PROJECT_ROOT / "supabase-schema.sql").read_text(encoding="utf-8").lower()
config_js = (PROJECT_ROOT / "supabase-config.js").read_text(encoding="utf-8").lower()
checks = {
    "RLS ativado na tabela de progresso": "enable row level security" in schema_sql,
    "Bucket de materiais configurado como privado": "'study-materials', 'study-materials', false" in schema_sql,
    "Política de leitura do Storage limitada ao bucket": "bucket_id = 'study-materials'" in schema_sql,
    "Configuração sem chave service_role": "service_role" not in config_js,
    "Configuração sem chave sb_secret_": "sb_secret_" not in config_js,
}
for label, passed in checks.items():
    print(f"{'OK' if passed else 'FALHOU'} — {label}")
assert all(checks.values()), "Revise configuração e políticas antes de publicar."

## 5. Desenvolvimento, testes e operação

Não há etapa de compilação do frontend. Use o Python instalado e um servidor HTTP estático; abrir a aplicação com `file://` pode impedir requisições ao JSON.

```powershell
python extractor.py
python -m http.server 8000
```

Acesse `http://localhost:8000`. O extrator usa a biblioteca padrão; execute os testes na raiz do repositório:

```powershell
python -m unittest discover -s tests
```

### Atualizar materiais e publicar

1. Atualize localmente os arquivos Markdown de origem.
2. Rode `python extractor.py` e revise contagens, questões não identificadas e gabaritos no ambiente privado apropriado.
3. Envie manifesto, índice e `subjects/*.json` ao bucket privado; não faça commit desses arquivos.
4. Teste login, carregamento, navegação e sincronização localmente.
5. Confira `git status` e `.gitignore`; publique apenas código/documentação necessários.

A extração é heurística. Marcadores fora do padrão podem manter questões na teoria ou impedir o reconhecimento de respostas, assunto e banca. Flashcards são sugestões. Confirme gabaritos na fonte original e respeite a privacidade/direitos dos materiais.

A célula executa a suíte sem imprimir textos das questões de teste.

In [ ]:
import importlib.util
import io
import unittest

test_file = PROJECT_ROOT / "tests" / "test_extractor.py"
spec = importlib.util.spec_from_file_location("project_test_extractor", test_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Não foi possível carregar a suíte: {test_file}")
test_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_module)
suite = unittest.defaultTestLoader.loadTestsFromModule(test_module)
stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=0).run(suite)
print(stream.getvalue().strip())
print(f"Testes executados: {result.testsRun}; erros: {len(result.errors)}; falhas: {len(result.failures)}")
assert result.wasSuccessful(), "A suíte do projeto apresentou falhas; consulte o resultado acima."

## 6. Diagnóstico e manutenção

| Sintoma | O que conferir |
|---|---|
| JSON local não carrega | Sirva a pasta por HTTP; não use `file://`. |
| Login falha | URL/chave publishable, usuário permitido, URL de retorno e configuração Auth do Supabase. |
| Materiais não carregam | Bucket privado, caminhos do manifesto, JSONs enviados e política de leitura. |
| Histórico não sincroniza | Conexão, sessão e RLS para `user_progress`; o progresso local permanece no navegador. |
| Questão/gabarito não reconhecido | Marcador numerado, alternativas e forma de apresentação do gabarito; conferir fonte original. |
| Interface antiga após publicação | Carregamento de JS/CSS e parâmetros de versão dos arquivos no HTML para invalidar cache. |
| Testes não encontrados | Executar `python -m unittest discover -s tests` a partir da raiz. |

### Princípios de manutenção

- Preserve o JSON como contrato entre extrator e interface; atualize gerador, consumidor e testes juntos.
- Mantenha erros de autenticação/rede explícitos; não os transforme em conteúdo vazio com aparência de sucesso.
- Ao trocar a conta autorizada, revise cliente, SQL/RLS e políticas implantadas.
- Evite dependências desnecessárias: frontend nativo e Python padrão.

### Resumo

O projeto combina geração local de materiais, aplicação estática no navegador e serviços gerenciados pelo Supabase. O extrator define o formato consumido pelo frontend; os testes protegem as etapas de limpeza e extração; políticas RLS/Storage impõem o controle de acesso. Materiais didáticos permanecem privados e não pertencem ao repositório público.